# LUAD Step 5: 301bp LUAD Model Training

This notebook trains mutation-aware deep learning models on the 301bp LUAD patient dataset.

It compares three model families:

- LSTM
- BiLSTM
- CNN

The binary stage target is `LABEL_STAGE_BINARY`.

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Bidirectional, Conv1D, Dense, Dropout, GlobalMaxPooling1D, Input, LSTM, MaxPooling1D
from tensorflow.keras.models import Model

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
tf.keras.utils.set_random_seed(42)

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "luad_processed").exists() and (PROJECT_DIR / "luad_processeed_data").exists():
    DATA_DIR = PROJECT_DIR / "luad_processeed_data"
else:
    DATA_DIR = PROJECT_DIR / "luad_processed"

RESULTS_DIR = PROJECT_DIR / "model_results"
INPUT_PATH = DATA_DIR / "final_luad_301bp_patient_dataset.csv"
RESULTS_CSV = RESULTS_DIR / "05_301bp_luad_model_training_results.csv"
PREDICTIONS_CSV = RESULTS_DIR / "05_301bp_luad_model_training_predictions.csv"
SUMMARY_PATH = RESULTS_DIR / "05_301bp_luad_model_training_summary.txt"

TARGET_WINDOW_LENGTH = 301
TARGET_CENTER = TARGET_WINDOW_LENGTH // 2
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20
BATCH_SIZE = 64
EPOCHS = 12

print(f"Project directory: {PROJECT_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Input dataset: {INPUT_PATH}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Target window length: {TARGET_WINDOW_LENGTH}")
print(f"Target mutation index: {TARGET_CENTER}")

In [ ]:
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Missing input dataset: {INPUT_PATH}")

patient_df = pd.read_csv(INPUT_PATH, dtype=str, low_memory=False)
patient_df.columns = [str(column).strip().upper() for column in patient_df.columns]

required_columns = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "SEX",
    "SEX_ENCODED",
    "LABEL_STAGE_BINARY",
    "MUTATED_GENE_LIST",
    "MUTATED_DNA_301_LIST",
    "OBSERVED_MASK_301_LIST",
    "MUTATION_MASK_301_LIST",
    "MUTATION_WINDOW_COUNT",
]
missing_columns = sorted(set(required_columns) - set(patient_df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if patient_df.duplicated(["COHORT", "PATIENT_ID"]).any():
    raise ValueError("Input dataset is not unique at COHORT + PATIENT_ID level.")

for column in ["SEX_ENCODED", "LABEL_STAGE_BINARY", "MUTATION_WINDOW_COUNT"]:
    patient_df[column] = pd.to_numeric(patient_df[column], errors="coerce")

print("Patient-level rows:", len(patient_df))
print("Cohorts:", patient_df["COHORT"].nunique())
print("Binary stage distribution:")
display(patient_df["LABEL_STAGE_BINARY"].value_counts(dropna=False).sort_index())

In [ ]:
def clean_text(value: object) -> str:
    return "" if pd.isna(value) else str(value).strip()


def parse_json_list(value: object) -> list:
    text = clean_text(value)
    if not text:
        return []
    parsed = json.loads(text)
    if not isinstance(parsed, list):
        raise ValueError("Expected a JSON list")
    return parsed


DNA_TO_INDEX = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4}


def one_hot_encode(sequence: str) -> np.ndarray:
    encoded = np.zeros((TARGET_WINDOW_LENGTH, 5), dtype=np.float32)
    text = clean_text(sequence).upper()
    for index, base in enumerate(text[:TARGET_WINDOW_LENGTH]):
        encoded[index, DNA_TO_INDEX.get(base, 4)] = 1.0
    return encoded


mutation_rows: List[Dict[str, object]] = []
for _, patient in patient_df.iterrows():
    genes = parse_json_list(patient["MUTATED_GENE_LIST"])
    sequences = parse_json_list(patient["MUTATED_DNA_301_LIST"])
    observed_masks = parse_json_list(patient["OBSERVED_MASK_301_LIST"])
    mutation_masks = parse_json_list(patient["MUTATION_MASK_301_LIST"])
    if not (len(genes) == len(sequences) == len(observed_masks) == len(mutation_masks)):
        raise ValueError(f"List length mismatch for {patient['COHORT']} / {patient['PATIENT_ID']}")
    weight = 1.0 / max(1, int(patient["MUTATION_WINDOW_COUNT"]))
    for window_index, (gene, sequence, observed_mask, mutation_mask) in enumerate(zip(genes, sequences, observed_masks, mutation_masks)):
        mutation_rows.append(
            {
                "COHORT": patient["COHORT"],
                "SAMPLE_ID": patient["SAMPLE_ID"],
                "PATIENT_ID": patient["PATIENT_ID"],
                "WINDOW_INDEX": window_index,
                "GENE_SYMBOL": gene,
                "DNA_301": clean_text(sequence),
                "OBSERVED_MASK_301": observed_mask,
                "MUTATION_MASK_301": mutation_mask,
                "LABEL_STAGE_BINARY": int(float(patient["LABEL_STAGE_BINARY"])),
                "SEX_ENCODED": int(float(patient["SEX_ENCODED"])),
                "WINDOW_WEIGHT": weight,
            }
        )

window_df = pd.DataFrame(mutation_rows)
window_df["LABEL_STAGE_BINARY"] = window_df["LABEL_STAGE_BINARY"].astype(int)

print("Mutation-level rows:", len(window_df))
print("Unique patients represented:", window_df["PATIENT_ID"].nunique())
print("Average windows per patient:", round(len(window_df) / window_df["PATIENT_ID"].nunique(), 2))

In [ ]:
def validate_301(sequence: str) -> bool:
    text = clean_text(sequence).upper()
    return len(text) == TARGET_WINDOW_LENGTH and bool(re.fullmatch(r"[ACGTN]+", text))


window_df["DNA_301_VALID"] = window_df["DNA_301"].map(validate_301)
window_df["OBSERVED_MASK_SUM"] = window_df["OBSERVED_MASK_301"].map(lambda mask: sum(int(v) for v in mask))
window_df["MUTATION_MASK_SUM"] = window_df["MUTATION_MASK_301"].map(lambda mask: sum(int(v) for v in mask))

print("All 301bp sequences valid:", bool(window_df["DNA_301_VALID"].all()))
print("All mutation masks sum to 1:", bool((window_df["MUTATION_MASK_SUM"] == 1).all()))
print("Average observed bases per window:", round(window_df["OBSERVED_MASK_SUM"].mean(), 2))
display(window_df[["COHORT", "PATIENT_ID", "GENE_SYMBOL", "DNA_301_VALID", "OBSERVED_MASK_SUM", "MUTATION_MASK_SUM"]].head())

In [ ]:
patient_labels = patient_df[["COHORT", "PATIENT_ID", "LABEL_STAGE_BINARY"]].copy()
train_patients, test_patients = train_test_split(
    patient_labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=patient_labels["LABEL_STAGE_BINARY"],
)
train_patients, val_patients = train_test_split(
    train_patients,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=train_patients["LABEL_STAGE_BINARY"],
)

train_ids = set(train_patients["PATIENT_ID"])
val_ids = set(val_patients["PATIENT_ID"])
test_ids = set(test_patients["PATIENT_ID"])
if train_ids & val_ids or train_ids & test_ids or val_ids & test_ids:
    raise ValueError("Patient-level split overlap detected.")

train_windows = window_df[window_df["PATIENT_ID"].isin(train_ids)].copy()
val_windows = window_df[window_df["PATIENT_ID"].isin(val_ids)].copy()
test_windows = window_df[window_df["PATIENT_ID"].isin(test_ids)].copy()

def stack_inputs(frame: pd.DataFrame) -> np.ndarray:
    sequence_tensor = np.stack(frame["DNA_301"].map(one_hot_encode).to_list())
    observed_tensor = np.asarray(frame["OBSERVED_MASK_301"].to_list(), dtype=np.float32)[..., np.newaxis]
    mutation_tensor = np.asarray(frame["MUTATION_MASK_301"].to_list(), dtype=np.float32)[..., np.newaxis]
    return np.concatenate([sequence_tensor, observed_tensor, mutation_tensor], axis=-1)


X_train = stack_inputs(train_windows)
X_val = stack_inputs(val_windows)
X_test = stack_inputs(test_windows)
y_train = train_windows["LABEL_STAGE_BINARY"].to_numpy(dtype=np.float32)
y_val = val_windows["LABEL_STAGE_BINARY"].to_numpy(dtype=np.float32)
y_test = test_windows["LABEL_STAGE_BINARY"].to_numpy(dtype=np.float32)

class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_train.astype(int))
class_weight_map = {0: float(class_weights[0]), 1: float(class_weights[1])}
train_weights = train_windows["WINDOW_WEIGHT"].to_numpy(dtype=np.float32) * np.vectorize(class_weight_map.get)(y_train.astype(int))
val_weights = val_windows["WINDOW_WEIGHT"].to_numpy(dtype=np.float32) * np.vectorize(class_weight_map.get)(y_val.astype(int))

print("Train windows:", len(train_windows))
print("Validation windows:", len(val_windows))
print("Test windows:", len(test_windows))
print("Input tensor shape:", X_train.shape)
print("Class weights:", class_weight_map)

In [ ]:
def build_lstm_model(input_shape: Tuple[int, int]) -> Model:
    inputs = Input(shape=input_shape, name="mutation_aware_input")
    x = LSTM(96, dropout=0.25, name="dna_lstm")(inputs)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.30)(x)
    outputs = Dense(1, activation="sigmoid", name="prediction")(x)
    model = Model(inputs=inputs, outputs=outputs, name="mutation_aware_lstm")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return model


def build_bilstm_model(input_shape: Tuple[int, int]) -> Model:
    inputs = Input(shape=input_shape, name="mutation_aware_input")
    x = Bidirectional(LSTM(64, dropout=0.25), name="dna_bilstm")(inputs)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.30)(x)
    outputs = Dense(1, activation="sigmoid", name="prediction")(x)
    model = Model(inputs=inputs, outputs=outputs, name="mutation_aware_bilstm")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return model


def build_cnn_model(input_shape: Tuple[int, int]) -> Model:
    inputs = Input(shape=input_shape, name="mutation_aware_input")
    x = Conv1D(64, 7, padding="same", activation="relu", name="dna_conv1")(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, 5, padding="same", activation="relu", name="dna_conv2")(x)
    x = BatchNormalization()(x)
    x = GlobalMaxPooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.30)(x)
    outputs = Dense(1, activation="sigmoid", name="prediction")(x)
    model = Model(inputs=inputs, outputs=outputs, name="mutation_aware_cnn")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    return model


def best_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    candidates = np.linspace(0.05, 0.95, 91)
    best_score = -1.0
    best_value = 0.5
    for threshold in candidates:
        prediction = (y_prob >= threshold).astype(int)
        score = f1_score(y_true, prediction, zero_division=0)
        if score > best_score:
            best_score = score
            best_value = float(threshold)
    return best_value


def evaluate_predictions(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, object]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "auc": float(roc_auc_score(y_true, y_prob)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "classification_report": classification_report(y_true, y_pred, target_names=["Early", "Advanced"], zero_division=0),
        "predictions": y_pred,
    }


def fit_and_score(model_name: str, model_builder) -> Dict[str, object]:
    tf.keras.backend.clear_session()
    model = model_builder(X_train.shape[1:])
    callbacks = [EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True), ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2, min_lr=1e-5)]
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val, val_weights), sample_weight=train_weights, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2, callbacks=callbacks)
    val_prob = model.predict(X_val, verbose=0).ravel()
    threshold = best_threshold(y_val, val_prob)
    test_prob = model.predict(X_test, verbose=0).ravel()
    test_metrics = evaluate_predictions(y_test, test_prob, threshold)
    val_metrics = evaluate_predictions(y_val, val_prob, threshold)
    return {
        "model": model_name,
        "threshold": threshold,
        "epochs_ran": len(history.history.get("loss", [])),
        "val_auc": val_metrics["auc"],
        "val_f1": val_metrics["f1"],
        "test_auc": test_metrics["auc"],
        "test_accuracy": test_metrics["accuracy"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "confusion_matrix": test_metrics["confusion_matrix"],
        "classification_report": test_metrics["classification_report"],
        "test_prob": test_prob,
        "test_pred": test_metrics["predictions"],
    }

print("Model builders ready.")

In [ ]:
results: List[Dict[str, object]] = []
for model_name, builder in [("LSTM", build_lstm_model), ("BiLSTM", build_bilstm_model), ("CNN", build_cnn_model)]:
    print(f"Training {model_name}...")
    results.append(fit_and_score(model_name, builder))

results_df = pd.DataFrame([{k: v for k, v in result.items() if k not in {"test_prob", "test_pred", "classification_report", "confusion_matrix"}} for result in results]).sort_values(["test_auc", "test_f1"], ascending=False).reset_index(drop=True)
display(results_df)

best_row = results_df.iloc[0]
best_result = next(result for result in results if result["model"] == best_row["model"])
print(f"Best model: {best_row['model']}")
print("Threshold:", best_result["threshold"])
print("Confusion matrix:")
print(np.array(best_result["confusion_matrix"]))
print()
print(best_result["classification_report"])

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_CSV, index=False)

predictions_df = test_windows[["COHORT", "PATIENT_ID", "SAMPLE_ID", "WINDOW_INDEX", "GENE_SYMBOL", "LABEL_STAGE_BINARY"]].copy()
predictions_df["predicted_probability"] = best_result["test_prob"]
predictions_df["predicted_label"] = best_result["test_pred"]
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

summary_lines = [
    "LUAD 301bp model training summary",
    f"Input dataset: {INPUT_PATH.resolve()}",
    f"Total patients: {len(patient_df)}",
    f"Total windows: {len(window_df)}",
    f"Train windows: {len(train_windows)}",
    f"Validation windows: {len(val_windows)}",
    f"Test windows: {len(test_windows)}",
    f"Input tensor shape: {X_train.shape}",
    "",
    "Ablation results (sorted by held-out AUC):",
]
for _, row in results_df.iterrows():
    summary_lines.append(f"- {row['model']}: threshold={row['threshold']:.2f}, test_auc={row['test_auc']:.4f}, test_f1={row['test_f1']:.4f}, test_bal_acc={row['test_balanced_accuracy']:.4f}")
summary_lines.extend(["", f"Best model: {best_row['model']}", f"Best confusion matrix: {best_result['confusion_matrix']}", f"Results CSV: {RESULTS_CSV.resolve()}", f"Predictions CSV: {PREDICTIONS_CSV.resolve()}"])
SUMMARY_PATH.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print(f"Saved results CSV: {RESULTS_CSV}")
print(f"Saved predictions CSV: {PREDICTIONS_CSV}")
print(f"Saved summary: {SUMMARY_PATH}")

## Notes

This notebook assumes `final_luad_301bp_patient_dataset.csv` has been created by the previous dataset-construction notebook.

If you want, I can also create a companion notebook for threshold plots, ROC curves, or a clinical-only baseline.